# Zonaprop Scraper
Scraper de departamentos en **venta** y **alquiler** en Capital Federal.

- Usa **Playwright** (Chromium headless) para las páginas de listado (JS-rendered)
- Usa **requests en paralelo** (ThreadPoolExecutor) para las fichas de detalle
- Produce `df_venta` y `df_alquiler` con schema compatible con Argenprop


## 1. Instalación de dependencias

## 2. Instalar navegador Chromium

Ejecutar esta celda una sola vez por entorno.

## 3. Imports de utilidades compartidas

In [1]:
import sys
import os
import re, time, json
import requests
import asyncio
import nest_asyncio
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from playwright.async_api import async_playwright

# Configuración para Windows + Playwright + Jupyter
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
nest_asyncio.apply()

sys.path.append(os.path.abspath(".."))

from utils import HEADERS, SCHEMA_COLS, FEATURE_COLS, clean_text, parse_address, parse_price, extract_smart_features, build_output_df, save_df

## 4. URLs y funciones de Zonaprop

In [2]:
BASE_VENTA    = 'https://www.zonaprop.com.ar/departamentos-venta.html'
BASE_ALQUILER = 'https://www.zonaprop.com.ar/departamentos-alquiler.html'

# Paginacion: departamentos-venta-pagina-2.html, -pagina-3.html, ...

In [3]:
def _generate_urls(base_url, count):
    """Genera la lista de URLs paginadas para Zonaprop."""
    base = re.sub(r'\.html$', '', base_url)
    base = re.sub(r'-pagina-\d+', '', base)
    return [
        f'{base}.html' if i == 1 else f'{base}-pagina-{i}.html'
        for i in range(1, count + 1)
    ]


def _extract_aviso_info(soup):
    """
    Intenta parsear generalFeatures del objeto JS 'avisoInfo' embebido en la página.
    Si falla, cae en la lista de íconos del HTML.
    """
    for script in soup.find_all('script'):
        src_txt = script.string or ''
        if 'avisoInfo' not in src_txt or 'generalFeatures' not in src_txt:
            continue
        m = re.search(r"'generalFeatures':\s*(\{.*?\})\s*,\s*\n", src_txt, re.DOTALL)
        if m:
            try:
                gf     = json.loads(m.group(1))
                labels = []
                for _cat, items in gf.items():
                    for _id, item in items.items():
                        label = item.get('label', '')
                        value = item.get('value')
                        labels.append(f'{label}: {value}' if value else label)
                return ' | '.join(labels)
            except Exception:
                pass
        break

    # Fallback: lista de íconos renderizada en el HTML
    feat_ul = soup.find('ul', id='section-icon-features-property')
    if feat_ul:
        parts = [clean_text(li.get_text(' ', strip=True))
                 for li in feat_ul.find_all('li')]
        return ' | '.join(p for p in parts if p and p != 'N/A')
    return 'N/A'


def _zonaprop_detail(url):
    """
    Descarga la página de detalle de un aviso de Zonaprop con requests.
    Extrae descripción, características, fecha de publicación y vistas.
    """
    clean_url = url.split('?')[0]
    out = {
        "Descripción":     "Sin descripción",
        "Caracteristicas": "N/A",
        "Publicado":       "N/A",
        "Visualizaciones": None,
    }
    try:
        r    = requests.get(clean_url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(r.content, 'lxml')

        # Descripción larga
        long_desc = soup.find('div', id='longDescription')
        if long_desc:
            txt = clean_text(long_desc.get_text(' ', strip=True))
            txt = re.sub(r'Leer descripci[oó]n completa.*', '', txt,
                         flags=re.IGNORECASE).strip()
            if len(txt) > 20:
                out["Descripción"] = txt

        # Características (JS object → fallback HTML icons)
        out["Caracteristicas"] = _extract_aviso_info(soup)

        # Publicado + visualizaciones
        user_views = soup.find('div', id='user-views')
        if user_views:
            for p in user_views.find_all('p'):
                text = p.get_text(strip=True)
                if 'Publicado hace' in text:
                    out["Publicado"] = text
                elif 'visualizaciones' in text:
                    m = re.search(r'\d+', text)
                    out["Visualizaciones"] = int(m.group()) if m else None

    except Exception as e:
        print(f'    [detail error] {e}')
    return out


async def scrape_zonaprop(base_url: str, mode: str, max_pages: int = 5):
    """
    Scraper principal de Zonaprop.
      - Playwright para las páginas de listado (JS-rendered)
      - requests en paralelo para las páginas de detalle

    base_url  : URL base (sin paginación).
    mode      : 'venta' o 'alquiler'.
    max_pages : cantidad máxima de páginas de listado.

    Devuelve
    -------
    DataFrame con SCHEMA_COLS + FEATURE_COLS, o None si no hay datos.
    """

    all_records = []
    seen_ids    = set()
    target_urls = _generate_urls(base_url, max_pages)

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=True,
            args=['--disable-blink-features=AutomationControlled'],
        )
        context   = await browser.new_context(user_agent=HEADERS['User-Agent'])
        list_page = await context.new_page()
        # Bloquear recursos pesados para acelerar la carga
        await list_page.route('**/*.{png,jpg,jpeg,webp,woff,woff2}',
                              lambda r: r.abort())

        for url in target_urls:
            print(f'\n--- Listado [{mode.upper()}]: {url} ---')
            try:
                await list_page.goto(url, wait_until='domcontentloaded', timeout=60_000)
                # Scroll suave para activar lazy-loading
                for _ in range(3):
                    await list_page.mouse.wheel(0, 2000)
                    await list_page.wait_for_timeout(800)
                await list_page.wait_for_selector(
                    "div[data-qa='posting PROPERTY']", timeout=15_000
                )
                cards = await list_page.query_selector_all(
                    "div[data-qa='posting PROPERTY']"
                )

                # ── Recolectar datos de tarjetas ──────────────────────────
                page_cards = []
                for card in cards:
                    try:
                        link_el = await card.query_selector(
                            "h2[data-qa='POSTING_CARD_DESCRIPTION'] a"
                        )
                        if not link_el:
                            continue
                        href = await link_el.get_attribute('href') or ''
                        id_m = re.search(r'-(\d+)\.html', href)
                        lid  = id_m.group(1) if id_m else href
                        if lid in seen_ids:
                            continue

                        full_url = (
                            f'https://www.zonaprop.com.ar{href}'
                            if not href.startswith('http') else href
                        )

                        price_el  = await card.query_selector("[data-qa='POSTING_CARD_PRICE']")
                        exp_el    = await card.query_selector("[data-qa='expensas']")
                        precio, _ = parse_price(await price_el.inner_text() if price_el else '')
                        expensas  = clean_text(await exp_el.inner_text()) if exp_el else 'N/A'

                        addr_el = (
                            await card.query_selector(
                                'h4.postingLocations-module__location-address'
                            ) or await card.query_selector("[data-qa='POSTING_CARD_LOCATION']")
                        )
                        calle, altura, piso = parse_address(
                            clean_text(await addr_el.inner_text()) if addr_el else 'N/A'
                        )

                        loc_el    = await card.query_selector("[data-qa='POSTING_CARD_LOCATION']")
                        ubicacion = clean_text(await loc_el.inner_text()) if loc_el else 'N/A'

                        feat_el  = await card.query_selector("[data-qa='POSTING_CARD_FEATURES']")
                        if feat_el:
                            spans = await feat_el.query_selector_all('span')
                            detalles_list = []
                            for s in spans:
                                detalles_list.append(await s.inner_text())
                            detalles = clean_text(' | '.join(detalles_list))
                        else:
                            detalles = 'N/A'

                        seen_ids.add(lid)
                        page_cards.append({
                            'Link': full_url, 'Precio': precio, 'Expensas': expensas,
                            'Calle': calle, 'Altura': altura, 'Piso': piso,
                            'Ubicacion': ubicacion, 'Detalles': detalles,
                        })
                    except Exception as e:
                        print(f'    [card error] {e}')

                if not page_cards:
                    print('Sin tarjetas nuevas. Fin.')
                    break

                # ── Descargar fichas en paralelo ──────────────────────────
                print(f'  Descargando {len(page_cards)} fichas (paralelo)...')
                detail_map = {}
                with ThreadPoolExecutor(max_workers=10) as pool:
                    futs = {pool.submit(_zonaprop_detail, c['Link']): c['Link']
                            for c in page_cards}
                    for fut in as_completed(futs):
                        lnk = futs[fut]
                        try:
                            detail_map[lnk] = fut.result()
                        except Exception as e:
                            print(f'    [thread error] {e}')
                            detail_map[lnk] = {}

                for card in page_cards:
                    d = detail_map.get(card['Link'], {})
                    all_records.append({
                        'Fuente':          'Zonaprop',
                        'Precio':          card['Precio'],
                        'Expensas':        card['Expensas'],
                        'Calle':           card['Calle'],
                        'Altura':          card['Altura'],
                        'Piso':            card['Piso'],
                        'Ubicacion':       card['Ubicacion'],
                        'Detalles':        card['Detalles'],
                        'Caracteristicas': d.get('Caracteristicas', 'N/A'),
                        'Descripción':     d.get('Descripción',     'Sin descripción'),
                        'Publicado':       d.get('Publicado',       'N/A'),
                        'Visualizaciones': d.get('Visualizaciones', None),
                        'Link':            card['Link'],
                    })

                print(f'  Nuevas: {len(page_cards)} | Total acumulado: {len(all_records)}')
                await context.clear_cookies()

            except Exception as e:
                print(f'❌ Error: {e}')
                continue

        await browser.close()

    return build_output_df(all_records)

print("✅ Funciones de Zonaprop listas.")

✅ Funciones de Zonaprop listas.


## 5. Scraping: Departamentos en Venta

In [4]:
df_venta = await scrape_zonaprop(BASE_VENTA, 'venta', max_pages=3)

if df_venta is not None:
    print(f'\nTotal venta: {len(df_venta)} propiedades')
    save_df(df_venta, 'zonaprop', 'venta')
    display(df_venta.head())

NotImplementedError: 

## 6. Scraping: Departamentos en Alquiler

In [ ]:
df_alquiler = await scrape_zonaprop(BASE_ALQUILER, 'alquiler', max_pages=3)

if df_alquiler is not None:
    print(f'\nTotal alquiler: {len(df_alquiler)} propiedades')
    save_df(df_alquiler, 'zonaprop', 'alquiler')
    display(df_alquiler.head())

## 7. Chequeo que se haya scrapeado bien

In [ ]:
# ── Resumen rápido ───────────────────────────────────────────────────────
for label, df in [('Venta', df_venta), ('Alquiler', df_alquiler)]:
    if df is not None:
        print(f'\n📊 {label} — {len(df)} propiedades')
        pct = (df[FEATURE_COLS].mean() * 100).round(1).astype(str) + '%'
        print(pct.to_string())